# Fetch Raw Data from Common Crawl

Thin notebook driver. Logic lives in `src/dataset/crawl/commoncrawl.py`.

See `docs/Patterns.md` — **Notebook Driver + src Utility**.

**Output:** `data/raw/documents.jsonl`, `data/raw/manifest.json`

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from dataset.crawl.commoncrawl import CrawlConfig, fetch_commoncrawl, list_available_crawls, pick_crawl, select_wet_paths
from dataset.paths import data_dir, resolve_repo_root

REPO_ROOT = resolve_repo_root(REPO_ROOT)
RAW_DIR = data_dir(REPO_ROOT) / "raw"

# --- config (adjust here) ---
CRAWL_ID = "CC-MAIN-2024-46"
NUM_WET_FILES = 2
MAX_RECORDS = 5_000
MIN_TEXT_CHARS = 200
SEED = 42

config = CrawlConfig(
    crawl_id=CRAWL_ID,
    num_wet_files=NUM_WET_FILES,
    max_records=MAX_RECORDS,
    min_text_chars=MIN_TEXT_CHARS,
    seed=SEED,
    output_jsonl=RAW_DIR / "documents.jsonl",
    manifest_path=RAW_DIR / "manifest.json",
)

print(f"Repo root:   {REPO_ROOT}")
print(f"Output dir:  {RAW_DIR}")
print(f"Crawl:       {config.crawl_id}")
print(f"WET files:   {config.num_wet_files}")
print(f"Max records: {config.max_records:,}")

## Inspect available crawls

In [ ]:
crawl_info = pick_crawl(config.crawl_id)
config.crawl_id = crawl_info["id"]

print("Available crawls (latest 5):")
for crawl in list_available_crawls()[:5]:
    print(f"  {crawl['id']}  ({crawl.get('name', 'n/a')})")
print()
print(f"Selected: {crawl_info['id']}")
print(f"Name:     {crawl_info.get('name')}")

selected_paths = select_wet_paths(config.crawl_id, config.num_wet_files, config.seed)
print(f"\nWET shards to fetch ({len(selected_paths)}):")
for path in selected_paths:
    print(f"  {path}")

## Fetch and save

WET files are streamed in memory — nothing cached to disk.

In [ ]:
result = fetch_commoncrawl(config)

print(f"Saved {result.records_written:,} records")
print(f"  -> {result.output_jsonl}")
print(f"  -> {result.manifest_path}")

## Review output

In [ ]:
print(json.dumps(result.manifest, indent=2))

with open(result.output_jsonl, encoding="utf-8") as fh:
    sample = json.loads(fh.readline())

print("\nSample record:")
print(f"  url:        {sample['url']}")
print(f"  char_count: {sample['char_count']}")
print(f"  text[:200]: {sample['text'][:200]!r}...")